In [6]:
import os
from dotenv import load_dotenv
from pprint import pprint

In [7]:
from langchain.chat_models import init_chat_model
from langchain_openai import  ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI
from langchain.messages import  HumanMessage,AIMessage,SystemMessage

load_dotenv()

True

In [8]:
model_name="gemini-2.5-flash"
model_provider="google_genai"


api_key = os.environ.get("GOOGLE_API_KEY") 
google_model_google_api = GoogleGenerativeAI(model=model_name, api_key=api_key)
google_model_universal_api = init_chat_model(model=model_name, model_provider=model_provider)

In [9]:
local_gemini4_llm = ChatOpenAI(
 base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio", # LM Studio accepts any string for the API key
    model="google/gemma-4-e4b" )

In [10]:
model = local_gemini4_llm

In [11]:
from langchain.messages import HumanMessage

response = model.invoke(
   "What is good man?"
    )

In [12]:
response

AIMessage(content='Because "good" is a word that means something different to every culture, philosophy, and individual person, there is no single definition or checklist for what makes a good man.\n\nHowever, if we look at common threads in literature, philosophy, psychology, and human relationships, certain recurring themes emerge. Generally speaking, being a "good man" is less about achieving a perfect state and more about committing to a **process of growth, accountability, and empathy.**\n\nHere is an exploration of the concept across three major dimensions: **Character (Internal), Behavior (Action),** and **Relationship (External).**\n\n***\n\n### 🧠 I. The Internal Compass: Character and Mindset\n\nA good man starts with work being done internally—with his own thoughts, biases, and motivations. This is about integrity first.\n\n*   **Integrity:** This is the most crucial trait. It means that your actions align with your stated values. A man of integrity does the right thing when 

In [13]:
pprint(response.content)

('Because "good" is a word that means something different to every culture, '
 'philosophy, and individual person, there is no single definition or '
 'checklist for what makes a good man.\n'
 '\n'
 'However, if we look at common threads in literature, philosophy, psychology, '
 'and human relationships, certain recurring themes emerge. Generally '
 'speaking, being a "good man" is less about achieving a perfect state and '
 'more about committing to a **process of growth, accountability, and '
 'empathy.**\n'
 '\n'
 'Here is an exploration of the concept across three major dimensions: '
 '**Character (Internal), Behavior (Action),** and **Relationship '
 '(External).**\n'
 '\n'
 '***\n'
 '\n'
 '### 🧠 I. The Internal Compass: Character and Mindset\n'
 '\n'
 'A good man starts with work being done internally—with his own thoughts, '
 'biases, and motivations. This is about integrity first.\n'
 '\n'
 '*   **Integrity:** This is the most crucial trait. It means that your '
 'actions align

In [14]:
response.response_metadata

{'token_usage': {'completion_tokens': 1000,
  'prompt_tokens': 21,
  'total_tokens': 1021,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 457,
   'rejected_prediction_tokens': None},
  'prompt_tokens_details': None},
 'model_provider': 'openai',
 'model_name': 'google/gemma-4-e4b',
 'system_fingerprint': 'google/gemma-4-e4b',
 'id': 'chatcmpl-rnjnumcllkcisy7g6iidxn',
 'finish_reason': 'length',
 'logprobs': None}

#testing agents

In [15]:
from langchain.agents import create_agent

agent=create_agent(model=model)


In [16]:
response = agent.invoke(
    {
        "messages":[HumanMessage(content="What is todays date?")]
    }
    )

In [17]:
response

{'messages': [HumanMessage(content='What is todays date?', additional_kwargs={}, response_metadata={}, id='f1a37857-fc8b-4ae5-9938-6829b179d164'),
  AIMessage(content="I do not have access to real-time information, including today's current date. Please check a clock or calendar on your device for the precise date!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 238, 'prompt_tokens': 21, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 202, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'google/gemma-4-e4b', 'system_fingerprint': 'google/gemma-4-e4b', 'id': 'chatcmpl-tvlwkb4szzlt65cr80x0k', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7f2a-e8f9-7850-8456-72998f0273e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 238, 'total_tokens': 25

In [18]:
response['messages'][-1].content

"I do not have access to real-time information, including today's current date. Please check a clock or calendar on your device for the precise date!"

- A simple Agent with system prompt , structure output and user prompt. 
- The agent will be able to answer questions based on the system prompt and user prompt.
- The agent will be able to use the structure output to format the answer.
- The goal of the agent is to provide direction to complete a task.

In [19]:
#   agent goal.
#   system prompt
#   user prompt
#   structute output


In [20]:
system_prompt = '''
You are DirectionAgent, an AI designed to give clear, actionable direction for completing tasks.

Your responsibilities:
1. Understand the user’s goal.
2. Break the goal into steps.
3. Provide guidance, constraints, and next actions.
4. Use the structured output format exactly as defined.
5. Never invent details not provided by the user.

Your style:
- Direct, concise, and practical.
- Steps must be specific and executable.
- Avoid vague advice.

'''

In [24]:
system_prompt = SystemMessage(content="""
You are DirectionAgent. Your job is to break down tasks into clear, actionable steps.
Always respond using the DirectionOutput Pydantic model structure.
""")

In [25]:
from pydantic import BaseModel, Field
from typing import List

class DirectionOutput(BaseModel):
    goal: str = Field(..., description="Short restatement of the user’s goal")
    key_requirements: List[str] = Field(default_factory=list)
    step_by_step_plan: List[str] = Field(default_factory=list)
    risks_and_constraints: List[str] = Field(default_factory=list)
    next_action: str = Field(..., description="The single next action the user should take")



In [27]:
AI_agent = create_agent(model=model, response_format=DirectionOutput)

In [44]:
intro_message = (
    "Welcome to your task breakdown agent. "
    "This agent breaks down your tasks into specific, executable steps for completion."
)

ask = "Tell me what you want to accomplish so I can guide you."

print(intro_message)

# Get user goal
goal = input(f"{ask}: ").strip()

# Basic validation
if not goal:
    print("You didn't enter a task. Please try again.")
else:
    question = HumanMessage(content=goal)
    response = AI_agent.invoke(
        {"messages": [system_prompt,question]},
    ) 

Welcome to your task breakdown agent. This agent breaks down your tasks into specific, executable steps for completion.


In [45]:
response 

{'messages': [SystemMessage(content='\nYou are DirectionAgent. Your job is to break down tasks into clear, actionable steps.\nAlways respond using the DirectionOutput Pydantic model structure.\n', additional_kwargs={}, response_metadata={}, id='cf0035d6-47b5-4eb2-ba5d-e47c9c6fff0c'),
  HumanMessage(content='weekly planner', additional_kwargs={}, response_metadata={}, id='53af6ac5-2f6b-42d0-b953-2df921b54460'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 383, 'prompt_tokens': 189, 'total_tokens': 572, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'google/gemma-4-e4b', 'system_fingerprint': 'google/gemma-4-e4b', 'id': 'chatcmpl-p4bcotm4q99257pz4nn54g', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e7f38-c23c-7613-8fae-af83b1

In [46]:
response['structured_response']

DirectionOutput(goal='Create a comprehensive and actionable weekly planning system.', key_requirements=['Structure the plan for maximum productivity.', 'Include dedicated time slots for tasks, breaks, and personal well-being.', 'Provide flexibility for unexpected events (buffer time).', 'Offer various tools/methods for different user preferences (digital, analog, printable).'], step_by_step_plan=['Step 1: Define Core Areas - Identify 3-5 critical life domains that need tracking (e.g., Work, Health, Family, Self-Development).', 'Step 2: Establish Routine Blocks - Block out non-negotiable times first (sleep, work hours, gym time, commute). These are the anchors.', 'Step 3: Time Blocking & Task Allocation - Schedule specific tasks into available blocks, assigning realistic time estimates for each item.', 'Step 4: Review and Buffer - At the start of the week, review all commitments. Add dedicated "buffer" or "flex" time (e.g., 1-2 hours) to absorb unexpected delays or urgent tasks.', 'Step

In [51]:
response['messages'][-1].content

'Returning structured response: goal=\'Create a comprehensive and actionable weekly planning system.\' key_requirements=[\'Structure the plan for maximum productivity.\', \'Include dedicated time slots for tasks, breaks, and personal well-being.\', \'Provide flexibility for unexpected events (buffer time).\', \'Offer various tools/methods for different user preferences (digital, analog, printable).\'] step_by_step_plan=[\'Step 1: Define Core Areas - Identify 3-5 critical life domains that need tracking (e.g., Work, Health, Family, Self-Development).\', \'Step 2: Establish Routine Blocks - Block out non-negotiable times first (sleep, work hours, gym time, commute). These are the anchors.\', \'Step 3: Time Blocking & Task Allocation - Schedule specific tasks into available blocks, assigning realistic time estimates for each item.\', \'Step 4: Review and Buffer - At the start of the week, review all commitments. Add dedicated "buffer" or "flex" time (e.g., 1-2 hours) to absorb unexpected 

In [58]:
parsed = response['structured_response']

In [60]:
parsed

DirectionOutput(goal='Create a comprehensive and actionable weekly planning system.', key_requirements=['Structure the plan for maximum productivity.', 'Include dedicated time slots for tasks, breaks, and personal well-being.', 'Provide flexibility for unexpected events (buffer time).', 'Offer various tools/methods for different user preferences (digital, analog, printable).'], step_by_step_plan=['Step 1: Define Core Areas - Identify 3-5 critical life domains that need tracking (e.g., Work, Health, Family, Self-Development).', 'Step 2: Establish Routine Blocks - Block out non-negotiable times first (sleep, work hours, gym time, commute). These are the anchors.', 'Step 3: Time Blocking & Task Allocation - Schedule specific tasks into available blocks, assigning realistic time estimates for each item.', 'Step 4: Review and Buffer - At the start of the week, review all commitments. Add dedicated "buffer" or "flex" time (e.g., 1-2 hours) to absorb unexpected delays or urgent tasks.', 'Step

In [61]:
parsed.next_action

'Tell me your preferred planning style (e.g., digital calendar like Google Calendar, physical bullet journal, or printed template) and the main areas you want to track (e.g., work projects, fitness goals, personal life, finances).'